# FootballPulse AI enrichment — Kaggle runner

Notebook enrich các bài bóng đá tiếng Anh bằng **Qwen3-0.6B** trên Kaggle.

Bản này được chia thành nhiều cell để dễ theo dõi/debug. Log được ghi **cả ra output của cell** và vào file `/kaggle/working/footballpulse-enrichment.log`.

## Thứ tự chạy

1. Chạy lần lượt các cell **1 → 9** để định nghĩa config/hàm.
2. Chạy cell **10 — Preflight**. Chỉ tiếp tục nếu thấy `preflight_completed`.
3. Chạy cell **11 — Main run** để xử lý batch.

## Output

- `/kaggle/working/footballpulse-enrichment.log` — log chi tiết.
- `/kaggle/working/results.jsonl` — kết quả từng article, flush sau mỗi article.
- `/kaggle/working/progress.json` — tiến độ gần nhất.
- `/kaggle/working/job-report.json` — báo cáo cuối batch.

Log bao gồm: function start/end/error, input validation, chunking, prompt/tokenization, model generation, JSON parse/repair, claim filtering/evidence recovery, ghi file, RAM/disk, VRAM/GPU, ETA và traceback khi lỗi.


## 1. Imports và cấu hình


In [1]:
"""FootballPulse Kaggle Qwen batch runner configuration."""

from __future__ import annotations

import hashlib
import json
import logging
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from functools import wraps
from pathlib import Path
from typing import Any, Callable, TypeVar, ParamSpec

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

LOG_PATH = OUTPUT_ROOT / "footballpulse-enrichment.log"
PROGRESS_PATH = OUTPUT_ROOT / "progress.json"
RESULTS_PATH = OUTPUT_ROOT / "results.jsonl"
REPORT_PATH = OUTPUT_ROOT / "job-report.json"

PROMPT_VERSION = "article-enrichment-v1"
MIN_TRANSFORMERS_VERSION = (4, 51, 0)

MAX_NEW_TOKENS = 512
MAX_INPUT_TOKENS = 4_096
MAX_CHUNK_WORDS = 1_200
CHUNK_OVERLAP_WORDS = 150

RESOURCE_LOG_EVERY_ARTICLES = 10
VALIDATION_LOG_EVERY_ROWS = 1_000


# Dataset resolution:
# Khi một .ipynb được import sang Kaggle notebook mới, các "Input" cũ có thể
# không còn được attach. Vì vậy notebook:
#   1) ưu tiên dataset đang có dưới /kaggle/input;
#   2) nếu không có thì tự resolve bằng kagglehub.dataset_download().
DATASET_HANDLE = os.getenv(
    "FOOTBALLPULSE_DATASET_HANDLE",
    "pmv259/footballpulse-ai-batches",
).strip()
ALLOW_KAGGLEHUB_DATASET_FALLBACK = (
    os.getenv("FOOTBALLPULSE_ALLOW_DATASET_DOWNLOAD", "1").strip().lower()
    not in {"0", "false", "no", "off"}
)

# Model resolution:
# - Ưu tiên model đã được Kaggle mount local.
# - Nếu không thấy model local, fallback qua kagglehub.model_download()
#   bằng chính model_version trong manifest.
MODEL_HANDLE_OVERRIDE = os.getenv("FOOTBALLPULSE_MODEL_HANDLE", "").strip()
ALLOW_KAGGLEHUB_MODEL_FALLBACK = (
    os.getenv("FOOTBALLPULSE_ALLOW_MODEL_DOWNLOAD", "1").strip().lower()
    not in {"0", "false", "no", "off"}
)

# Có thể đổi thành DEBUG nếu muốn log sâu hơn:
# os.environ["FOOTBALLPULSE_LOG_LEVEL"] = "DEBUG"
LOG_LEVEL_NAME = os.getenv("FOOTBALLPULSE_LOG_LEVEL", "INFO").upper()
LOG_LEVEL = getattr(logging, LOG_LEVEL_NAME, logging.INFO)

SESSION_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "-"
    + uuid.uuid4().hex[:8]
)

REQUIRED_RESULT_FIELDS = frozenset({"event_type", "summary_en", "claims"})
REQUIRED_MANIFEST_FIELDS = frozenset(
    {"batch_id", "article_count", "articles_sha256", "model_version", "prompt_version"}
)
REQUIRED_ARTICLE_FIELDS = frozenset(
    {"article_version_id", "input_hash", "cleaned_content"}
)

ALLOWED_PREDICATES = frozenset(
    {
        "EXPRESSED_INTEREST",
        "CONTACTED",
        "SUBMITTED_BID",
        "ACCEPTED_BID",
        "REJECTED_BID",
        "COMPLETED_TRANSFER",
        "NEGOTIATING_CONTRACT",
        "SIGNED_CONTRACT",
        "SUFFERED_INJURY",
        "EXPECTED_RETURN",
        "MATCH_SCHEDULED",
        "MATCH_RESULT",
        "APPOINTED_COACH",
        "DISMISSED_COACH",
        "DENIED_REPORT",
    }
)

LOGGER = logging.getLogger("footballpulse.kaggle.runner")

P = ParamSpec("P")
R = TypeVar("R")

print(
    f"config_loaded session_id={SESSION_ID} "
    f"log_level={LOG_LEVEL_NAME} output_root={OUTPUT_ROOT}"
)


config_loaded session_id=20260817T155215Z-bb26cb6a log_level=INFO output_root=/kaggle/working


## 2. Logger nền tảng và function tracing


In [2]:
def configure_runner_logging() -> None:
    """Log cả stdout của Kaggle và file persistent trong /kaggle/working."""
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    handlers: list[logging.Handler] = [
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_PATH, mode="a", encoding="utf-8"),
    ]

    logging.basicConfig(
        level=LOG_LEVEL,
        format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
        handlers=handlers,
        force=True,
    )

    LOGGER.setLevel(LOG_LEVEL)
    LOGGER.info(
        "logger_configured session_id=%s log_level=%s log_path=%s",
        SESSION_ID,
        LOG_LEVEL_NAME,
        LOG_PATH,
    )


def _format_log_value(value: object) -> str:
    """Giữ log một dòng và tránh field quá dài."""
    text = str(value).replace("\n", "\\n").replace("\r", "\\r")
    if len(text) > 500:
        text = text[:497] + "..."
    return text


def log_event(
    event: str,
    *,
    level: int = logging.INFO,
    **fields: object,
) -> None:
    detail = " ".join(
        f"{name}={_format_log_value(value)}"
        for name, value in fields.items()
    )
    prefix = f"session_id={SESSION_ID}"
    LOGGER.log(level, "%s %s%s", event, prefix, f" {detail}" if detail else "")


def log_progress(event: str, **fields: object) -> None:
    """Alias cho các event INFO-level."""
    log_event(event, level=logging.INFO, **fields)


def logged_function(
    *,
    level: int = logging.INFO,
) -> Callable[[Callable[P, R]], Callable[P, R]]:
    """Decorator log function start/end/error + duration mà không dump args lớn."""
    def decorator(function: Callable[P, R]) -> Callable[P, R]:
        @wraps(function)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            started = time.monotonic()
            log_event(
                "function_started",
                level=level,
                function=function.__name__,
            )
            try:
                result = function(*args, **kwargs)
            except Exception as error:
                log_event(
                    "function_failed",
                    level=logging.ERROR,
                    function=function.__name__,
                    error_type=type(error).__name__,
                    error=str(error),
                    duration_seconds=round(time.monotonic() - started, 3),
                )
                raise

            log_event(
                "function_completed",
                level=level,
                function=function.__name__,
                duration_seconds=round(time.monotonic() - started, 3),
            )
            return result

        return wrapper

    return decorator


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def gib(value: int | float) -> float:
    return round(float(value) / (1024**3), 3)


def parse_semver_prefix(value: str) -> tuple[int, int, int]:
    match = re.match(r"^\s*(\d+)\.(\d+)\.(\d+)", value)
    if not match:
        return (0, 0, 0)
    return tuple(int(part) for part in match.groups())  # type: ignore[return-value]


configure_runner_logging()
log_progress("logging_helpers_ready")


2026-08-17 15:52:15,037 INFO [footballpulse.kaggle.runner] logger_configured session_id=20260817T155215Z-bb26cb6a log_level=INFO log_path=/kaggle/working/footballpulse-enrichment.log
2026-08-17 15:52:15,038 INFO [footballpulse.kaggle.runner] logging_helpers_ready session_id=20260817T155215Z-bb26cb6a


## 3. Resource monitor: RAM, disk, GPU/VRAM, môi trường


In [3]:
@logged_function()
def read_ram_stats() -> dict[str, object]:
    """Đọc RAM bằng psutil; fallback sang /proc/meminfo."""
    try:
        import psutil

        memory = psutil.virtual_memory()
        result = {
            "ram_used_gib": gib(memory.used),
            "ram_available_gib": gib(memory.available),
            "ram_total_gib": gib(memory.total),
            "ram_percent": round(float(memory.percent), 1),
            "ram_source": "psutil",
        }
        log_progress("ram_stats_read", **result)
        return result
    except Exception as psutil_error:
        log_event(
            "ram_psutil_unavailable",
            level=logging.WARNING,
            error_type=type(psutil_error).__name__,
        )

    try:
        values: dict[str, int] = {}
        for line in Path("/proc/meminfo").read_text(encoding="utf-8").splitlines():
            key, raw = line.split(":", 1)
            values[key] = int(raw.strip().split()[0]) * 1024

        total = values["MemTotal"]
        available = values["MemAvailable"]
        used = total - available
        result = {
            "ram_used_gib": gib(used),
            "ram_available_gib": gib(available),
            "ram_total_gib": gib(total),
            "ram_percent": round(used * 100 / total, 1),
            "ram_source": "proc_meminfo",
        }
        log_progress("ram_stats_read", **result)
        return result
    except Exception as error:
        log_event(
            "ram_stats_failed",
            level=logging.WARNING,
            error_type=type(error).__name__,
            error=str(error),
        )
        return {"ram_error": type(error).__name__}


@logged_function()
def log_resource_snapshot(label: str) -> None:
    """Log RAM, disk và CUDA allocator/VRAM."""
    fields = read_ram_stats()

    try:
        disk = shutil.disk_usage(OUTPUT_ROOT)
        fields.update(
            {
                "disk_used_gib": gib(disk.used),
                "disk_free_gib": gib(disk.free),
                "disk_total_gib": gib(disk.total),
            }
        )
    except OSError as error:
        fields["disk_error"] = type(error).__name__

    log_progress("resource_snapshot", label=label, **fields)

    try:
        import torch

        if not torch.cuda.is_available():
            log_event(
                "gpu_memory_skipped",
                level=logging.WARNING,
                label=label,
                reason="cuda_not_available",
            )
            return

        for index in range(torch.cuda.device_count()):
            free_bytes, total_bytes = torch.cuda.mem_get_info(index)
            log_progress(
                "gpu_memory",
                label=label,
                gpu=index,
                name=torch.cuda.get_device_name(index),
                free_gib=gib(free_bytes),
                total_gib=gib(total_bytes),
                allocated_gib=gib(torch.cuda.memory_allocated(index)),
                reserved_gib=gib(torch.cuda.memory_reserved(index)),
                max_allocated_gib=gib(torch.cuda.max_memory_allocated(index)),
            )
    except Exception as error:
        log_event(
            "gpu_memory_unavailable",
            level=logging.WARNING,
            label=label,
            error_type=type(error).__name__,
            error=str(error),
        )


@logged_function()
def log_nvidia_smi(label: str) -> None:
    """Log GPU utilization, nhiệt độ và power nếu nvidia-smi tồn tại."""
    command = [
        "nvidia-smi",
        "--query-gpu=index,name,memory.used,memory.total,"
        "utilization.gpu,temperature.gpu,power.draw",
        "--format=csv,noheader,nounits",
    ]

    log_progress("nvidia_smi_started", label=label)

    try:
        completed = subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
            timeout=10,
        )

        lines = [line.strip() for line in completed.stdout.splitlines() if line.strip()]
        log_progress("nvidia_smi_rows_received", label=label, row_count=len(lines))

        for line in lines:
            log_progress("nvidia_smi", label=label, values=line)
    except (OSError, subprocess.SubprocessError) as error:
        log_event(
            "nvidia_smi_unavailable",
            level=logging.WARNING,
            label=label,
            error_type=type(error).__name__,
            error=str(error),
        )


@logged_function()
def log_environment(*, require_gpu: bool = True) -> None:
    import torch
    import transformers

    version = str(transformers.__version__)
    log_progress(
        "runtime_environment",
        python=sys.version.split()[0],
        platform=platform.platform(),
        cpu_count=os.cpu_count(),
        torch=torch.__version__,
        transformers=version,
        cuda_runtime=torch.version.cuda,
        cuda_available=torch.cuda.is_available(),
        gpu_count=torch.cuda.device_count(),
    )

    parsed_version = parse_semver_prefix(version)
    log_progress(
        "transformers_version_check",
        actual=parsed_version,
        minimum=MIN_TRANSFORMERS_VERSION,
    )
    if parsed_version < MIN_TRANSFORMERS_VERSION:
        required = ".".join(str(value) for value in MIN_TRANSFORMERS_VERSION)
        raise RuntimeError(
            f"Qwen3 requires transformers>={required}, found {version}. "
            "Use a newer Kaggle environment or enable Internet and upgrade transformers."
        )

    if require_gpu and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU is required for this notebook, but torch.cuda.is_available() is False. "
            "In Kaggle Session options, select a GPU accelerator."
        )

    if torch.cuda.is_available():
        build_arch_list = list(torch.cuda.get_arch_list())
        supported_sm = sorted(
            arch
            for arch in build_arch_list
            if arch.startswith("sm_")
        )

        log_progress(
            "torch_cuda_arch_support",
            torch=torch.__version__,
            cuda_runtime=torch.version.cuda,
            supported_arches=",".join(supported_sm) or "unknown",
        )

        unsupported_devices: list[str] = []

        for index in range(torch.cuda.device_count()):
            major, minor = torch.cuda.get_device_capability(index)
            props = torch.cuda.get_device_properties(index)
            device_name = torch.cuda.get_device_name(index)
            device_arch = f"sm_{major}{minor}"

            # get_arch_list() reflects the CUDA architectures compiled into
            # the current PyTorch wheel. For this Kaggle environment this
            # catches Pascal/P100 incompatibility before torch kernels run.
            arch_supported = (
                not supported_sm
                or device_arch in supported_sm
            )

            log_progress(
                "cuda_device",
                gpu=index,
                name=device_name,
                compute_capability=f"{major}.{minor}",
                cuda_arch=device_arch,
                torch_arch_supported=arch_supported,
                total_memory_gib=gib(props.total_memory),
                multiprocessors=props.multi_processor_count,
            )

            if not arch_supported:
                unsupported_devices.append(
                    f"gpu={index} name={device_name} arch={device_arch}"
                )

        if unsupported_devices:
            details = "; ".join(unsupported_devices)
            raise RuntimeError(
                "The selected Kaggle GPU is not supported by the current "
                f"PyTorch build ({torch.__version__}, CUDA {torch.version.cuda}). "
                f"Unsupported device(s): {details}. "
                f"PyTorch build arches: {','.join(supported_sm) or 'unknown'}. "
                "For this notebook, switch Kaggle Accelerator from P100 to "
                "GPU T4 x2 (the notebook will use cuda:0)."
            )


## 4. Tìm input/model và validate batch


In [4]:
def _discover_batch_pairs(root: Path) -> list[tuple[Path, Path]]:
    """Find directories containing both manifest.json and articles.jsonl."""
    root = Path(root)

    log_progress(
        "batch_pair_scan_started",
        root=root,
        exists=root.exists(),
    )

    if not root.exists():
        return []

    pairs: list[tuple[Path, Path]] = []
    visited_dirs = 0

    try:
        for current_dir, dirnames, filenames in os.walk(root, followlinks=True):
            visited_dirs += 1

            dirnames[:] = [
                name
                for name in dirnames
                if name not in {
                    ".git",
                    "__pycache__",
                    ".ipynb_checkpoints",
                    ".cache",
                }
            ]

            names = set(filenames)
            if "manifest.json" in names and "articles.jsonl" in names:
                directory = Path(current_dir)
                pairs.append(
                    (
                        directory / "manifest.json",
                        directory / "articles.jsonl",
                    )
                )
    except OSError as error:
        log_event(
            "batch_pair_scan_failed",
            level=logging.WARNING,
            root=root,
            error_type=type(error).__name__,
            error=str(error),
        )

    unique = sorted(
        set(pairs),
        key=lambda pair: str(pair[0]),
    )

    log_progress(
        "batch_pair_scan_completed",
        root=root,
        visited_dirs=visited_dirs,
        count=len(unique),
        preview=";".join(str(pair[0].parent) for pair in unique[:10]) or "none",
    )

    return unique


@logged_function()
def find_batch_files(
    root: Path,
    dataset_handle: str | None = None,
) -> tuple[Path, Path]:
    """
    Resolve FootballPulse batch input.

    Resolution order:
      1. Existing Kaggle inputs under /kaggle/input.
      2. kagglehub.dataset_download(DATASET_HANDLE) fallback.

    This makes the notebook survive being imported into a new Kaggle notebook
    where the previous UI Input attachments are no longer present.
    """
    root = Path(root)
    requested_handle = (
        str(dataset_handle).strip()
        if dataset_handle
        else DATASET_HANDLE
    )

    log_progress(
        "input_batch_search_started",
        root=root,
        dataset_handle=requested_handle or "none",
        fallback_enabled=ALLOW_KAGGLEHUB_DATASET_FALLBACK,
    )

    try:
        top_level_entries = sorted(
            root.iterdir(),
            key=lambda p: p.name.casefold(),
        ) if root.exists() else []

        log_progress(
            "kaggle_input_entries",
            count=len(top_level_entries),
            preview=";".join(str(p) for p in top_level_entries[:20]) or "none",
        )
    except OSError as error:
        top_level_entries = []
        log_event(
            "kaggle_input_listing_failed",
            level=logging.WARNING,
            root=root,
            error_type=type(error).__name__,
            error=str(error),
        )

    local_candidates = _discover_batch_pairs(root)

    if len(local_candidates) == 1:
        manifest_path, articles_path = local_candidates[0]
        log_progress(
            "input_batch_resolution_selected",
            source="kaggle_input",
            manifest=manifest_path,
            articles=articles_path,
        )
        log_progress(
            "input_batch_found",
            manifest=manifest_path,
            articles=articles_path,
            articles_size_bytes=articles_path.stat().st_size,
        )
        return manifest_path, articles_path

    if len(local_candidates) > 1:
        preview = ";".join(
            str(item[0].parent)
            for item in local_candidates[:10]
        )
        raise RuntimeError(
            "multiple batch datasets were found under /kaggle/input; "
            "the notebook will not guess which one to use. "
            f"candidates={preview}"
        )

    # No local batch.
    if not requested_handle:
        raise RuntimeError(
            "no FootballPulse batch dataset was found under /kaggle/input, "
            "and FOOTBALLPULSE_DATASET_HANDLE is empty"
        )

    if not ALLOW_KAGGLEHUB_DATASET_FALLBACK:
        raise RuntimeError(
            "no FootballPulse batch dataset was found under /kaggle/input "
            "and KaggleHub dataset fallback is disabled. "
            f"Expected dataset handle: {requested_handle}"
        )

    log_progress(
        "kagglehub_dataset_resolution_started",
        handle=requested_handle,
    )

    try:
        import kagglehub
    except ImportError as error:
        raise RuntimeError(
            "kagglehub is not installed, so the notebook cannot resolve "
            f"dataset {requested_handle!r}"
        ) from error

    try:
        download_started = time.monotonic()
        resolved = kagglehub.dataset_download(requested_handle)
        download_duration = time.monotonic() - download_started
    except Exception as error:
        log_event(
            "kagglehub_dataset_resolution_failed",
            level=logging.ERROR,
            handle=requested_handle,
            error_type=type(error).__name__,
            error=str(error),
        )
        raise RuntimeError(
            "No batch dataset is attached to this Kaggle notebook and "
            "kagglehub.dataset_download() also failed. "
            f"handle={requested_handle!r}; error={error}"
        ) from error

    resolved_root = Path(resolved)

    log_progress(
        "kagglehub_dataset_resolution_completed",
        handle=requested_handle,
        path=resolved_root,
        duration_seconds=round(download_duration, 3),
    )

    downloaded_candidates = _discover_batch_pairs(resolved_root)

    if len(downloaded_candidates) != 1:
        preview = ";".join(
            str(item[0].parent)
            for item in downloaded_candidates[:10]
        ) or "none"
        raise RuntimeError(
            "KaggleHub resolved the dataset, but the notebook could not find "
            "exactly one directory containing manifest.json + articles.jsonl. "
            f"resolved_root={resolved_root}; candidates={preview}"
        )

    manifest_path, articles_path = downloaded_candidates[0]

    log_progress(
        "input_batch_resolution_selected",
        source="kagglehub",
        handle=requested_handle,
        manifest=manifest_path,
        articles=articles_path,
    )
    log_progress(
        "input_batch_found",
        manifest=manifest_path,
        articles=articles_path,
        articles_size_bytes=articles_path.stat().st_size,
    )

    return manifest_path, articles_path


@logged_function()
def validate_model_directory(model_path: Path) -> Path:
    """Validate một thư mục model Hugging Face/Qwen3 trước khi load."""
    model_path = Path(model_path).resolve()

    log_progress(
        "model_directory_validation_started",
        path=model_path,
    )

    if not model_path.is_dir():
        raise RuntimeError(f"model path is not a directory: {model_path}")

    config_path = model_path / "config.json"
    if not config_path.is_file():
        raise RuntimeError(f"model config.json not found in {model_path}")

    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise RuntimeError(
            f"cannot read model config.json from {config_path}: {error}"
        ) from error

    model_type = str(config.get("model_type", ""))
    architectures = config.get("architectures", [])

    if not model_type.casefold().startswith("qwen3"):
        raise RuntimeError(
            "resolved model is not Qwen3: "
            f"path={model_path} model_type={model_type!r} "
            f"architectures={architectures!r}"
        )

    weights = (
        list(model_path.glob("*.safetensors"))
        + list(model_path.glob("pytorch_model*.bin"))
    )

    # Một số HF model dùng index + nhiều shard safetensors.
    index_files = (
        list(model_path.glob("*.safetensors.index.json"))
        + list(model_path.glob("pytorch_model*.bin.index.json"))
    )

    if not weights and not index_files:
        raise RuntimeError(
            f"Qwen3 config found but no model weights were found in {model_path}"
        )

    model_bytes = sum(
        item.stat().st_size
        for item in model_path.rglob("*")
        if item.is_file()
    )

    log_progress(
        "model_directory_validation_completed",
        path=model_path,
        model_type=model_type,
        architectures=",".join(map(str, architectures)) if architectures else "unknown",
        weight_files=len(weights),
        index_files=len(index_files),
        model_dir_size_gib=gib(model_bytes),
    )
    return model_path


@logged_function()
def find_model_path(
    root: Path,
    model_handle: str | None = None,
) -> Path:
    """
    Resolve Qwen3 model theo 2 tầng:

    1. Tìm model đã được Kaggle mount trong /kaggle/input.
    2. Nếu không thấy, dùng kagglehub.model_download(model_handle).

    `model_handle` lấy từ manifest["model_version"], ví dụ:
    qwen-lm/qwen-3/transformers/0.6b/1
    """
    root = Path(root)
    requested_handle = (
        MODEL_HANDLE_OVERRIDE
        or (str(model_handle).strip() if model_handle else "")
    )

    log_progress(
        "model_search_started",
        root=root,
        requested_handle=requested_handle or "none",
        fallback_enabled=ALLOW_KAGGLEHUB_MODEL_FALLBACK,
    )

    # Log ngay các input đang thực sự được mount để debug Kaggle dễ hơn.
    try:
        top_level_entries = sorted(root.iterdir(), key=lambda p: p.name.casefold())
        log_progress(
            "kaggle_input_entries",
            count=len(top_level_entries),
            preview=";".join(str(p) for p in top_level_entries[:20]) or "none",
        )
    except OSError as error:
        log_event(
            "kaggle_input_listing_failed",
            level=logging.WARNING,
            root=root,
            error_type=type(error).__name__,
            error=str(error),
        )

    # Pathlib glob có thể bỏ sót một vài cấu trúc mount/symlink;
    # os.walk(..., followlinks=True) robust hơn cho Kaggle model inputs.
    config_paths: list[Path] = []
    visited_dirs = 0

    for current_dir, dirnames, filenames in os.walk(root, followlinks=True):
        visited_dirs += 1

        # Bỏ một số thư mục cache/ẩn không cần thiết nếu xuất hiện.
        dirnames[:] = [
            name for name in dirnames
            if name not in {".git", "__pycache__", ".ipynb_checkpoints"}
        ]

        if "config.json" in filenames:
            config_paths.append(Path(current_dir) / "config.json")

    log_progress(
        "model_config_candidates_scanned",
        count=len(config_paths),
        visited_dirs=visited_dirs,
        preview=";".join(str(p) for p in config_paths[:10]) or "none",
    )

    candidates: list[Path] = []
    for config_path in config_paths:
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as error:
            log_event(
                "model_config_skipped",
                level=logging.DEBUG,
                path=config_path,
                error_type=type(error).__name__,
                error=str(error),
            )
            continue

        model_type = str(config.get("model_type", ""))
        architectures = config.get("architectures", [])

        log_event(
            "model_config_inspected",
            level=logging.DEBUG,
            path=config_path,
            model_type=model_type,
            architectures=architectures,
        )

        if model_type.casefold().startswith("qwen3"):
            candidates.append(config_path.parent)

    unique = sorted(set(candidates))

    log_progress(
        "qwen3_local_model_candidates_detected",
        count=len(unique),
        preview=";".join(str(item) for item in unique[:10]) or "none",
    )

    if len(unique) == 1:
        log_progress(
            "model_resolution_selected",
            source="kaggle_input",
            path=unique[0],
        )
        return validate_model_directory(unique[0])

    if len(unique) > 1:
        # Nếu có nhiều Qwen3 model local thì thử khớp handle theo slug/variation.
        handle_parts = [
            part.casefold()
            for part in requested_handle.split("/")
            if part
        ]

        scored: list[tuple[int, Path]] = []
        for candidate in unique:
            candidate_text = str(candidate).casefold()
            score = sum(
                1
                for part in handle_parts
                if part not in {"transformers", "1"}
                and part in candidate_text
            )
            scored.append((score, candidate))

        scored.sort(key=lambda item: (-item[0], str(item[1])))
        best_score = scored[0][0] if scored else 0
        best = [path for score, path in scored if score == best_score]

        log_progress(
            "multiple_local_models_scored",
            best_score=best_score,
            best_count=len(best),
            best_preview=";".join(str(p) for p in best[:5]) or "none",
        )

        if best_score > 0 and len(best) == 1:
            log_progress(
                "model_resolution_selected",
                source="kaggle_input_scored",
                path=best[0],
            )
            return validate_model_directory(best[0])

        raise RuntimeError(
            "multiple Qwen3 models were found in /kaggle/input and the notebook "
            "could not choose one safely. "
            f"requested_handle={requested_handle!r}; "
            f"candidates={';'.join(str(x) for x in unique[:10])}"
        )

    # Không có model local. Đây chính là trường hợp notebook trước của bạn gặp:
    # /kaggle/input chỉ có dataset, dù model xuất hiện trong UI.
    if not requested_handle:
        raise RuntimeError(
            "no Qwen3 model was found under /kaggle/input, and no model handle "
            "was provided by manifest or FOOTBALLPULSE_MODEL_HANDLE"
        )

    if not ALLOW_KAGGLEHUB_MODEL_FALLBACK:
        raise RuntimeError(
            "no Qwen3 model was found under /kaggle/input and KaggleHub fallback "
            "is disabled. Set FOOTBALLPULSE_ALLOW_MODEL_DOWNLOAD=1 or attach "
            f"the model explicitly. requested_handle={requested_handle}"
        )

    log_progress(
        "kagglehub_model_resolution_started",
        handle=requested_handle,
    )

    try:
        import kagglehub
    except ImportError as error:
        raise RuntimeError(
            "kagglehub is not installed, so the notebook cannot resolve "
            f"Kaggle model {requested_handle!r}"
        ) from error

    try:
        download_started = time.monotonic()
        resolved = kagglehub.model_download(requested_handle)
        download_duration = time.monotonic() - download_started
    except Exception as error:
        log_event(
            "kagglehub_model_resolution_failed",
            level=logging.ERROR,
            handle=requested_handle,
            error_type=type(error).__name__,
            error=str(error),
        )
        raise RuntimeError(
            "Kaggle model was not mounted locally and kagglehub.model_download "
            f"also failed for handle={requested_handle!r}: {error}"
        ) from error

    resolved_path = Path(resolved)

    log_progress(
        "kagglehub_model_resolution_completed",
        handle=requested_handle,
        path=resolved_path,
        duration_seconds=round(download_duration, 3),
    )

    log_progress(
        "model_resolution_selected",
        source="kagglehub",
        path=resolved_path,
    )

    return validate_model_directory(resolved_path)


@logged_function()
def load_manifest(manifest_path: Path) -> dict[str, Any]:
    log_progress(
        "manifest_load_started",
        path=manifest_path,
        size_bytes=manifest_path.stat().st_size,
    )

    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as error:
        raise RuntimeError(
            f"manifest.json is invalid JSON: line={error.lineno} column={error.colno} "
            f"message={error.msg}"
        ) from error

    if not isinstance(manifest, dict):
        raise RuntimeError("manifest.json must contain one JSON object")

    log_progress(
        "manifest_json_loaded",
        field_count=len(manifest),
        fields=",".join(sorted(manifest.keys())),
    )

    missing = REQUIRED_MANIFEST_FIELDS - manifest.keys()
    if missing:
        raise RuntimeError(f"manifest.json is missing fields: {sorted(missing)}")

    log_progress(
        "manifest_required_fields_valid",
        batch_id=manifest.get("batch_id"),
        article_count=manifest.get("article_count"),
        model_version=manifest.get("model_version"),
        prompt_version=manifest.get("prompt_version"),
    )

    if manifest["prompt_version"] != PROMPT_VERSION:
        raise RuntimeError(
            "manifest prompt_version does not match this notebook: "
            f"manifest={manifest['prompt_version']} notebook={PROMPT_VERSION}"
        )

    if not isinstance(manifest["article_count"], int) or manifest["article_count"] < 0:
        raise RuntimeError("manifest article_count must be a non-negative integer")

    log_progress("manifest_validation_completed", batch_id=manifest["batch_id"])
    return manifest


@logged_function()
def validate_articles_file(articles_path: Path, expected_count: int) -> int:
    log_progress(
        "articles_validation_started",
        path=articles_path,
        expected_count=expected_count,
        size_bytes=articles_path.stat().st_size,
    )

    count = 0
    with articles_path.open(encoding="utf-8") as source:
        for line_number, line in enumerate(source, start=1):
            if not line.strip():
                raise RuntimeError(
                    f"articles.jsonl contains a blank line at line {line_number}"
                )

            try:
                article = json.loads(line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "articles.jsonl contains invalid JSON at "
                    f"line {line_number}: {error.msg}"
                ) from error

            if not isinstance(article, dict):
                raise RuntimeError(
                    f"articles.jsonl line {line_number} must be a JSON object"
                )

            missing = REQUIRED_ARTICLE_FIELDS - article.keys()
            if missing:
                raise RuntimeError(
                    f"articles.jsonl line {line_number} is missing fields: {sorted(missing)}"
                )

            if not isinstance(article["cleaned_content"], str):
                raise RuntimeError(
                    f"articles.jsonl line {line_number} cleaned_content must be a string"
                )

            count += 1

            if (
                count == 1
                or count % VALIDATION_LOG_EVERY_ROWS == 0
                or count == expected_count
            ):
                log_progress(
                    "articles_validation_progress",
                    validated=count,
                    expected=expected_count,
                    article_version_id=article.get("article_version_id"),
                    content_chars=len(article["cleaned_content"]),
                    canonical_entity_count=len(article.get("canonical_entities", []) or []),
                )

    if count != expected_count:
        raise RuntimeError(
            f"manifest article_count={expected_count}, "
            f"but articles.jsonl contains {count} rows"
        )

    log_progress("articles_validation_completed", validated_count=count)
    return count


@logged_function()
def validate_batch_checksum(
    articles_path: Path,
    manifest: dict[str, Any],
) -> str:
    expected = str(manifest["articles_sha256"])
    log_progress(
        "checksum_validation_started",
        path=articles_path,
        algorithm="sha256",
        expected=expected,
    )

    digest = hashlib.sha256()
    bytes_read = 0

    with articles_path.open("rb") as source:
        while True:
            block = source.read(1024 * 1024)
            if not block:
                break
            digest.update(block)
            bytes_read += len(block)

    actual = digest.hexdigest()
    log_progress(
        "checksum_computed",
        bytes_read=bytes_read,
        actual=actual,
    )

    if actual != expected:
        raise RuntimeError(
            "articles.jsonl checksum does not match manifest: "
            f"actual={actual} expected={expected}"
        )

    log_progress("checksum_validation_completed", sha256=actual)
    return actual


@logged_function()
def preflight() -> dict[str, Any]:
    configure_runner_logging()
    preflight_started = time.monotonic()

    log_progress(
        "preflight_started",
        input_root=INPUT_ROOT,
        output_root=OUTPUT_ROOT,
    )

    log_progress("preflight_stage_started", stage="environment")
    log_environment(require_gpu=True)
    log_resource_snapshot("preflight")
    log_nvidia_smi("preflight")
    log_progress("preflight_stage_completed", stage="environment")

    log_progress("preflight_stage_started", stage="input_discovery")
    manifest_path, articles_path = find_batch_files(
        INPUT_ROOT,
        DATASET_HANDLE,
    )
    log_progress("preflight_stage_completed", stage="input_discovery")

    log_progress("preflight_stage_started", stage="manifest")
    manifest = load_manifest(manifest_path)
    log_progress("preflight_stage_completed", stage="manifest")

    log_progress("preflight_stage_started", stage="checksum")
    articles_sha256 = validate_batch_checksum(articles_path, manifest)
    log_progress("preflight_stage_completed", stage="checksum")

    log_progress("preflight_stage_started", stage="articles_validation")
    row_count = validate_articles_file(
        articles_path,
        manifest["article_count"],
    )
    log_progress("preflight_stage_completed", stage="articles_validation")

    log_progress("preflight_stage_started", stage="model_discovery")
    model_path = find_model_path(
        INPUT_ROOT,
        manifest["model_version"],
    )
    log_progress("preflight_stage_completed", stage="model_discovery")

    log_progress(
        "preflight_completed",
        batch_id=manifest["batch_id"],
        article_count=row_count,
        articles_size_mib=round(articles_path.stat().st_size / (1024**2), 3),
        articles_sha256=articles_sha256,
        model_version=manifest["model_version"],
        prompt_version=manifest["prompt_version"],
        manifest=manifest_path,
        articles=articles_path,
        model=model_path,
        log_path=LOG_PATH,
        duration_seconds=round(time.monotonic() - preflight_started, 3),
    )

    return {
        "manifest_path": manifest_path,
        "articles_path": articles_path,
        "manifest": manifest,
        "articles_sha256": articles_sha256,
        "model_path": model_path,
    }


## 5. Parse JSON, chunking và tạo prompt


In [5]:
@logged_function()
def parse_model_json(text: str) -> dict[str, Any]:
    log_progress(
        "model_json_parse_started",
        raw_chars=len(text),
        starts_with=text.strip()[:40],
    )

    cleaned = re.sub(
        r"^\s*```(?:json)?|```\s*$",
        "",
        text.strip(),
        flags=re.IGNORECASE,
    )

    if cleaned != text.strip():
        log_progress(
            "model_json_markdown_fence_removed",
            cleaned_chars=len(cleaned),
        )

    try:
        value = json.loads(cleaned)
        parse_mode = "full_text"
    except json.JSONDecodeError as first_error:
        log_event(
            "model_json_full_parse_failed",
            level=logging.WARNING,
            line=first_error.lineno,
            column=first_error.colno,
            message=first_error.msg,
        )

        start = cleaned.find("{")
        end = cleaned.rfind("}")

        if start < 0 or end <= start:
            raise ValueError("model did not return a JSON object") from None

        log_progress(
            "model_json_object_slice_attempt",
            start=start,
            end=end,
            slice_chars=end - start + 1,
        )
        value = json.loads(cleaned[start : end + 1])
        parse_mode = "object_slice"

    if not isinstance(value, dict):
        raise ValueError("model output must be a JSON object")

    missing = REQUIRED_RESULT_FIELDS - value.keys()
    if missing:
        raise ValueError(
            f"model JSON is missing required result fields: {sorted(missing)}"
        )

    if not isinstance(value["summary_en"], str) or not value["summary_en"].strip():
        raise ValueError("model summary_en must be non-empty")

    if not isinstance(value["claims"], list):
        raise ValueError("model claims must be a list")

    log_progress(
        "model_json_parse_completed",
        parse_mode=parse_mode,
        event_type=value.get("event_type"),
        summary_chars=len(value["summary_en"]),
        claim_count=len(value["claims"]),
    )
    return value


@logged_function()
def content_chunks(
    content: str,
    *,
    max_words: int = MAX_CHUNK_WORDS,
    overlap_words: int = CHUNK_OVERLAP_WORDS,
) -> list[tuple[int, str]]:
    log_progress(
        "chunking_started",
        content_chars=len(content),
        max_words=max_words,
        overlap_words=overlap_words,
    )

    if max_words < 1 or overlap_words < 0 or overlap_words >= max_words:
        raise ValueError("invalid chunk limits")

    words = list(re.finditer(r"\S+", content))
    log_progress("chunking_words_counted", word_count=len(words))

    chunks: list[tuple[int, str]] = []
    first = 0

    while first < len(words):
        last = min(first + max_words, len(words))
        start = words[first].start()
        end = words[last - 1].end()

        chunk_text = content[start:end]
        chunks.append((start, chunk_text))

        log_progress(
            "chunk_created",
            chunk_number=len(chunks),
            first_word_index=first,
            last_word_index_exclusive=last,
            start_char=start,
            end_char=end,
            chars=len(chunk_text),
            words=last - first,
        )

        if len(chunks) > 64:
            raise ValueError("article requires more than 64 model chunks")

        if last == len(words):
            break

        next_first = last - overlap_words
        log_event(
            "chunk_overlap_applied",
            level=logging.DEBUG,
            current_last=last,
            next_first=next_first,
            overlap_words=overlap_words,
        )
        first = next_first

    log_progress(
        "chunking_completed",
        chunk_count=len(chunks),
        word_count=len(words),
    )
    return chunks


@logged_function()
def prompt_for(
    article: dict[str, Any],
    *,
    repair_text: str | None = None,
) -> str:
    article_id = article.get("article_version_id")
    canonical_entities = article.get("canonical_entities") or []

    log_progress(
        "prompt_build_started",
        article_version_id=article_id,
        content_chars=len(article.get("cleaned_content", "")),
        canonical_entity_count=len(canonical_entities),
        repair_mode=repair_text is not None,
    )

    if not canonical_entities:
        prompt = (
            "Return only one concise grounded English summary of this football article. "
            "Do not add facts, labels, JSON, markdown, or commentary.\nArticle:\n"
            + json.dumps(
                {
                    "title": article.get("title"),
                    "cleaned_content": article["cleaned_content"],
                },
                ensure_ascii=False,
            )
        )

        log_progress(
            "prompt_build_completed",
            article_version_id=article_id,
            mode="summary_only",
            prompt_chars=len(prompt),
        )
        return prompt

    claim_contract = [
        {
            "subject_entity_id": "UUID from canonical_entities",
            "predicate": "|".join(sorted(ALLOWED_PREDICATES)),
            "object_entity_id": "UUID or null",
            "object_text": "text or null; exactly one object form",
            "qualifiers": {
                "amount": None,
                "currency": None,
                "date": None,
                "injury": None,
                "score": None,
            },
            "certainty": "RUMOR|REPORTED|CONFIRMED|DENIED",
            "evidence_quote": "exact substring of cleaned_content",
            "evidence_start": 0,
            "evidence_end": 1,
        }
    ]

    contract = {
        "event_type": (
            "TRANSFER|CONTRACT|INJURY|MATCH|MANAGERIAL|DISCIPLINARY|OTHER"
        ),
        "summary_en": "grounded English summary",
        "claims": claim_contract,
    }

    repair = (
        "\nYour previous output was invalid. Return one corrected JSON object only:\n"
        + repair_text[:4_000]
        if repair_text
        else ""
    )

    prompt = (
        "Extract only facts supported by the English article. Never invent entities, "
        "numbers, dates, scores, or certainty. Evidence offsets use Python character "
        "indexes into cleaned_content. Return JSON only.\nSchema:\n"
        + json.dumps(contract, ensure_ascii=False)
        + "\nInput:\n"
        + json.dumps(article, ensure_ascii=False)
        + repair
    )

    log_progress(
        "prompt_build_completed",
        article_version_id=article_id,
        mode="structured_extraction",
        prompt_chars=len(prompt),
        repair_chars=len(repair),
    )
    return prompt


## 6. Load model và generation


In [6]:
@logged_function()
def load_model(model_path: Path) -> tuple[Any, Any]:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required to load Qwen3 in this notebook")

    started = time.monotonic()
    device = torch.device("cuda:0")
    dtype = torch.float16

    log_progress(
        "model_load_selected",
        device=device,
        dtype=str(dtype),
        model_path=model_path,
        note="Qwen3-0.6B runs on cuda:0; GPU 1 is intentionally left unused",
    )

    log_resource_snapshot("before_model_load")

    tokenizer_started = time.monotonic()
    log_progress("tokenizer_loading_started", model_path=model_path)
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        local_files_only=True,
    )
    log_progress(
        "tokenizer_loading_completed",
        duration_seconds=round(time.monotonic() - tokenizer_started, 3),
        tokenizer_class=type(tokenizer).__name__,
        vocab_size=getattr(tokenizer, "vocab_size", "unknown"),
    )

    model_started = time.monotonic()
    log_progress("model_weights_loading_started", model_path=model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        local_files_only=True,
        torch_dtype=dtype,
    )
    log_progress(
        "model_weights_loaded_to_cpu",
        duration_seconds=round(time.monotonic() - model_started, 3),
        model_class=type(model).__name__,
    )

    move_started = time.monotonic()
    log_progress("model_move_to_gpu_started", device=device)
    model.to(device)
    model.eval()
    log_progress(
        "model_move_to_gpu_completed",
        duration_seconds=round(time.monotonic() - move_started, 3),
        device=next(model.parameters()).device,
    )

    # Greedy decoding để extraction deterministic hơn.
    model.generation_config.do_sample = False

    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    trainable_count = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    log_progress(
        "model_configuration_ready",
        parameter_count=parameter_count,
        trainable_parameter_count=trainable_count,
        do_sample=model.generation_config.do_sample,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    log_resource_snapshot("after_model_load")
    log_nvidia_smi("after_model_load")

    log_progress(
        "model_ready",
        duration_seconds=round(time.monotonic() - started, 3),
    )
    return tokenizer, model


@logged_function()
def generate(tokenizer: Any, model: Any, prompt: str) -> str:
    import torch

    log_progress(
        "generation_pipeline_started",
        prompt_chars=len(prompt),
        max_input_tokens=MAX_INPUT_TOKENS,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    messages = [
        {
            "role": "system",
            "content": "You are a strict football fact extraction engine.",
        },
        {"role": "user", "content": prompt},
    ]

    template_started = time.monotonic()
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    log_progress(
        "chat_template_rendered",
        rendered_chars=len(rendered),
        duration_seconds=round(time.monotonic() - template_started, 3),
        thinking_enabled=False,
    )

    tokenization_started = time.monotonic()
    inputs = tokenizer(
        rendered,
        return_tensors="pt",
        truncation=False,
    )
    input_tokens = int(inputs["input_ids"].shape[1])
    log_progress(
        "prompt_tokenized",
        input_tokens=input_tokens,
        token_budget_remaining=MAX_INPUT_TOKENS - input_tokens,
        duration_seconds=round(time.monotonic() - tokenization_started, 3),
    )

    if input_tokens > MAX_INPUT_TOKENS:
        log_event(
            "prompt_token_limit_exceeded",
            level=logging.ERROR,
            input_tokens=input_tokens,
            max_input_tokens=MAX_INPUT_TOKENS,
        )
        raise ValueError(
            f"rendered prompt has {input_tokens} tokens, exceeding "
            f"MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}; "
            "reduce MAX_CHUNK_WORDS or article metadata"
        )

    device = next(model.parameters()).device
    log_progress("generation_device_selected", device=device)

    transfer_started = time.monotonic()
    inputs = inputs.to(device)
    log_event(
        "generation_inputs_moved",
        level=logging.DEBUG,
        device=device,
        duration_seconds=round(time.monotonic() - transfer_started, 3),
    )

    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
        log_event(
            "cuda_peak_memory_reset",
            level=logging.DEBUG,
            device=device,
        )

    generation_started = time.monotonic()
    log_progress(
        "model_generate_started",
        input_tokens=input_tokens,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    generation_duration = time.monotonic() - generation_started
    prompt_tokens = int(inputs["input_ids"].shape[1])
    output_tokens = int(generated.shape[1] - prompt_tokens)
    tokens_per_second = (
        round(output_tokens / generation_duration, 2)
        if generation_duration > 0
        else 0.0
    )

    log_progress(
        "model_generate_completed",
        input_tokens=prompt_tokens,
        output_tokens=output_tokens,
        duration_seconds=round(generation_duration, 3),
        output_tokens_per_second=tokens_per_second,
        gpu_peak_allocated_gib=(
            gib(torch.cuda.max_memory_allocated(device))
            if device.type == "cuda"
            else "n/a"
        ),
    )

    decode_started = time.monotonic()
    decoded = tokenizer.decode(
        generated[0][prompt_tokens:],
        skip_special_tokens=True,
    )
    log_progress(
        "generation_decode_completed",
        output_chars=len(decoded),
        duration_seconds=round(time.monotonic() - decode_started, 3),
    )

    return decoded


## 7. Extraction, claim validation và ghép kết quả article


In [7]:
@logged_function()
def extract_chunk(
    tokenizer: Any,
    model: Any,
    article: dict[str, Any],
) -> dict[str, Any]:
    article_id = article.get("article_version_id")
    canonical_entities = article.get("canonical_entities") or []

    log_progress(
        "chunk_extraction_started",
        article_version_id=article_id,
        content_chars=len(article.get("cleaned_content", "")),
        canonical_entity_count=len(canonical_entities),
    )

    prompt = prompt_for(article)
    raw = generate(tokenizer, model, prompt)

    log_progress(
        "chunk_model_output_received",
        article_version_id=article_id,
        raw_chars=len(raw),
    )

    if not canonical_entities:
        summary = raw.strip()[:4_000].strip()
        if not summary:
            raise ValueError("model summary_en must be non-empty")

        log_progress(
            "chunk_summary_only_completed",
            article_version_id=article_id,
            summary_chars=len(summary),
        )
        return {
            "event_type": "OTHER",
            "summary_en": summary,
            "claims": [],
        }

    try:
        parsed = parse_model_json(raw)
        log_progress(
            "chunk_json_parse_succeeded_first_try",
            article_version_id=article_id,
            claim_count=len(parsed["claims"]),
        )
        return parsed
    except (ValueError, json.JSONDecodeError) as first_error:
        log_event(
            "model_json_repair_started",
            level=logging.WARNING,
            article_version_id=article_id,
            error_type=type(first_error).__name__,
            error=str(first_error),
            raw_chars=len(raw),
        )

        repair_prompt = prompt_for(article, repair_text=raw)
        repaired = generate(tokenizer, model, repair_prompt)

        log_progress(
            "model_json_repair_output_received",
            article_version_id=article_id,
            repaired_chars=len(repaired),
        )

        parsed = parse_model_json(repaired)
        log_progress(
            "model_json_repair_completed",
            article_version_id=article_id,
            claim_count=len(parsed["claims"]),
        )
        return parsed


@logged_function()
def normalize_claim_evidence(
    claim: dict[str, Any],
    content: str,
) -> dict[str, Any]:
    quote = claim.get("evidence_quote")
    start = claim.get("evidence_start")
    end = claim.get("evidence_end")

    if not isinstance(quote, str) or not quote:
        raise ValueError("claim evidence quote must be non-empty")

    normalized = dict(claim)

    offsets_valid = (
        isinstance(start, int)
        and isinstance(end, int)
        and 0 <= start <= end <= len(content)
        and content[start:end] == quote
    )

    if offsets_valid:
        log_progress(
            "claim_evidence_offsets_valid",
            predicate=claim.get("predicate"),
            start=start,
            end=end,
            quote_chars=len(quote),
        )
        return normalized

    log_event(
        "claim_evidence_offsets_invalid_recovery_started",
        level=logging.WARNING,
        predicate=claim.get("predicate"),
        supplied_start=start,
        supplied_end=end,
        quote_chars=len(quote),
        content_chars=len(content),
    )

    recovered_start = content.find(quote)
    recovered_last = content.rfind(quote)

    if recovered_start < 0:
        raise ValueError("claim evidence quote is not present in cleaned_content")

    if recovered_start != recovered_last:
        raise ValueError(
            "claim evidence quote occurs multiple times; cannot recover unique offsets"
        )

    normalized["evidence_start"] = recovered_start
    normalized["evidence_end"] = recovered_start + len(quote)

    log_progress(
        "claim_evidence_offsets_recovered",
        predicate=claim.get("predicate"),
        recovered_start=normalized["evidence_start"],
        recovered_end=normalized["evidence_end"],
    )
    return normalized


def claim_grounding_problem(
    claim: dict[str, Any],
    canonical_ids: set[str],
) -> str | None:
    predicate = claim.get("predicate")
    if predicate not in ALLOWED_PREDICATES:
        return f"predicate_not_allowed:{predicate}"

    subject_entity_id = claim.get("subject_entity_id")
    if subject_entity_id not in canonical_ids:
        return f"subject_not_canonical:{subject_entity_id}"

    object_entity_id = claim.get("object_entity_id")
    if object_entity_id is not None and object_entity_id not in canonical_ids:
        return f"object_not_canonical:{object_entity_id}"

    return None


@logged_function()
def process_article(
    tokenizer: Any,
    model: Any,
    article: dict[str, Any],
    *,
    model_version: str,
    prompt_version: str,
) -> dict[str, Any]:
    article_started = time.monotonic()
    article_id = article["article_version_id"]

    canonical_entities = article.get("canonical_entities") or []
    canonical_ids = {
        str(entity["entity_id"])
        for entity in canonical_entities
        if isinstance(entity, dict) and "entity_id" in entity
    }

    log_progress(
        "article_processing_started",
        article_version_id=article_id,
        content_chars=len(article["cleaned_content"]),
        canonical_entity_count=len(canonical_entities),
        canonical_id_count=len(canonical_ids),
        unresolved_mention_count=len(article.get("unresolved_mentions", []) or []),
    )

    extracted_chunks: list[dict[str, Any]] = []
    global_claims: list[dict[str, Any]] = []

    chunks = content_chunks(article["cleaned_content"])
    log_progress(
        "article_chunking_completed",
        article_version_id=article_id,
        chunk_count=len(chunks),
    )

    for chunk_number, (start, chunk_text) in enumerate(chunks, start=1):
        chunk_started = time.monotonic()
        end = start + len(chunk_text)

        log_progress(
            "article_chunk_started",
            article_version_id=article_id,
            chunk_number=chunk_number,
            chunk_total=len(chunks),
            start_char=start,
            end_char=end,
            chunk_chars=len(chunk_text),
        )

        unresolved_mentions = article.get("unresolved_mentions", []) or []
        chunk_mentions: list[dict[str, Any]] = []

        for mention in unresolved_mentions:
            if not isinstance(mention, dict):
                log_event(
                    "unresolved_mention_skipped",
                    level=logging.WARNING,
                    article_version_id=article_id,
                    chunk_number=chunk_number,
                    reason="not_object",
                )
                continue

            mention_start = mention.get("start")
            mention_end = mention.get("end")

            if not isinstance(mention_start, int) or not isinstance(mention_end, int):
                log_event(
                    "unresolved_mention_skipped",
                    level=logging.WARNING,
                    article_version_id=article_id,
                    chunk_number=chunk_number,
                    reason="invalid_offsets",
                )
                continue

            if mention_start >= start and mention_end <= end:
                chunk_mentions.append(
                    {
                        **mention,
                        "start": mention_start - start,
                        "end": mention_end - start,
                    }
                )

        log_progress(
            "chunk_mentions_mapped",
            article_version_id=article_id,
            chunk_number=chunk_number,
            source_mentions=len(unresolved_mentions),
            mapped_mentions=len(chunk_mentions),
        )

        chunk_article = {
            **article,
            "cleaned_content": chunk_text,
            "unresolved_mentions": chunk_mentions,
        }

        extracted = extract_chunk(tokenizer, model, chunk_article)

        if "claims" not in extracted or not isinstance(extracted["claims"], list):
            raise ValueError("extracted chunk claims must be a list")

        log_progress(
            "article_chunk_extracted",
            article_version_id=article_id,
            chunk_number=chunk_number,
            event_type=extracted.get("event_type"),
            summary_chars=len(str(extracted.get("summary_en", ""))),
            claim_count=len(extracted["claims"]),
        )

        extracted_chunks.append(extracted)

        accepted_in_chunk = 0
        dropped_in_chunk = 0

        for claim_number, claim_value in enumerate(
            extracted["claims"],
            start=1,
        ):
            if not isinstance(claim_value, dict):
                raise ValueError("model claim must be an object")

            problem = claim_grounding_problem(
                claim_value,
                canonical_ids,
            )
            if problem is not None:
                dropped_in_chunk += 1
                log_event(
                    "claim_dropped_not_canonically_grounded",
                    level=logging.WARNING,
                    article_version_id=article_id,
                    chunk_number=chunk_number,
                    claim_number=claim_number,
                    predicate=claim_value.get("predicate"),
                    reason=problem,
                )
                continue

            claim = normalize_claim_evidence(
                claim_value,
                chunk_text,
            )

            local_start = claim["evidence_start"]
            local_end = claim["evidence_end"]

            claim["evidence_start"] = start + local_start
            claim["evidence_end"] = start + local_end
            global_claims.append(claim)
            accepted_in_chunk += 1

            log_progress(
                "claim_accepted",
                article_version_id=article_id,
                chunk_number=chunk_number,
                claim_number=claim_number,
                predicate=claim.get("predicate"),
                global_start=claim["evidence_start"],
                global_end=claim["evidence_end"],
            )

        log_progress(
            "article_chunk_completed",
            article_version_id=article_id,
            chunk_number=chunk_number,
            accepted_claims=accepted_in_chunk,
            dropped_claims=dropped_in_chunk,
            duration_seconds=round(time.monotonic() - chunk_started, 3),
        )

    if not extracted_chunks:
        raise ValueError("article content has no words")

    event_counts = Counter(
        str(item["event_type"])
        for item in extracted_chunks
    )
    event_type = max(
        event_counts,
        key=lambda value: event_counts[value],
    )

    log_progress(
        "article_event_type_aggregated",
        article_version_id=article_id,
        event_counts=dict(event_counts),
        selected_event_type=event_type,
    )

    summaries = list(
        dict.fromkeys(
            str(item["summary_en"]).strip()
            for item in extracted_chunks
        )
    )
    summary = " ".join(summaries)

    log_progress(
        "article_summary_aggregated",
        article_version_id=article_id,
        unique_summary_parts=len(summaries),
        summary_chars=len(summary),
        claim_count=len(global_claims),
    )

    if len(summary) > 4_000:
        raise ValueError(
            f"combined summary exceeds 4000 characters: {len(summary)}"
        )

    if len(global_claims) > 500:
        raise ValueError(
            f"combined claims exceed 500 items: {len(global_claims)}"
        )

    result = {
        "contract_version": "article-enrichment.v1",
        "article_version_id": article_id,
        "input_hash": article["input_hash"],
        "event_type": event_type,
        "summary_en": summary,
        "claims": global_claims,
        "model_version": model_version,
        "prompt_version": prompt_version,
    }

    log_progress(
        "article_processing_completed",
        article_version_id=article_id,
        event_type=event_type,
        claim_count=len(global_claims),
        summary_chars=len(summary),
        duration_seconds=round(time.monotonic() - article_started, 3),
    )

    return {
        "article_version_id": article_id,
        "status": "SUCCESS",
        "result": result,
    }


## 8. Error handling, progress và output


In [8]:
@logged_function()
def error_record(
    article: dict[str, Any],
    error: Exception,
) -> dict[str, Any]:
    record = {
        "article_version_id": article.get("article_version_id"),
        "input_hash": article.get("input_hash"),
        "status": "ERROR",
        "error_code": type(error).__name__.upper()[:80],
        "error": " ".join(str(error).split())[:500] or "unknown model error",
    }

    log_progress(
        "error_record_created",
        article_version_id=record["article_version_id"],
        error_code=record["error_code"],
    )
    return record


@logged_function()
def write_progress(
    *,
    manifest: dict[str, Any],
    processed_count: int,
    success_count: int,
    error_count: int,
    current_article_version_id: object,
    elapsed_seconds: float,
) -> None:
    total = manifest["article_count"]
    rate = processed_count / elapsed_seconds if elapsed_seconds > 0 else 0.0
    remaining = max(total - processed_count, 0)
    eta_seconds = remaining / rate if rate > 0 else None

    state = {
        "batch_id": manifest["batch_id"],
        "processed_count": processed_count,
        "article_count": total,
        "progress_percent": (
            round(processed_count * 100 / total, 2)
            if total
            else 100.0
        ),
        "success_count": success_count,
        "error_count": error_count,
        "current_article_version_id": current_article_version_id,
        "elapsed_seconds": round(elapsed_seconds, 2),
        "eta_seconds": (
            round(eta_seconds, 2)
            if eta_seconds is not None
            else None
        ),
        "updated_at": utc_now().isoformat(),
        "results_path": str(RESULTS_PATH),
        "log_path": str(LOG_PATH),
    }

    temporary = PROGRESS_PATH.with_suffix(".json.tmp")

    log_event(
        "progress_file_write_started",
        level=logging.DEBUG,
        temp_path=temporary,
        final_path=PROGRESS_PATH,
        processed_count=processed_count,
    )

    temporary.write_text(
        json.dumps(state, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(PROGRESS_PATH)

    log_progress(
        "progress_file_written",
        processed_count=processed_count,
        article_count=total,
        progress_percent=state["progress_percent"],
        success_count=success_count,
        error_count=error_count,
        eta_seconds=state["eta_seconds"],
        path=PROGRESS_PATH,
    )


@logged_function()
def maybe_recover_from_cuda_oom(error: Exception) -> None:
    if "out of memory" not in str(error).casefold():
        log_event(
            "cuda_oom_recovery_skipped",
            level=logging.DEBUG,
            reason="not_oom",
        )
        return

    log_event(
        "cuda_oom_detected",
        level=logging.ERROR,
        error=str(error),
    )

    try:
        import torch

        if torch.cuda.is_available():
            log_progress("cuda_oom_cache_clear_started")
            torch.cuda.empty_cache()
            log_progress("cuda_oom_cache_clear_completed")
            log_resource_snapshot("after_cuda_oom")
            log_nvidia_smi("after_cuda_oom")
    except Exception as cleanup_error:
        log_event(
            "cuda_oom_cache_clear_failed",
            level=logging.ERROR,
            error_type=type(cleanup_error).__name__,
            error=str(cleanup_error),
        )


@logged_function()
def write_job_report(
    *,
    manifest: dict[str, Any],
    articles_sha256: str,
    success_count: int,
    error_count: int,
    started_at: datetime,
) -> dict[str, Any]:
    report = {
        "contract_version": "ai-job-report.v1",
        "batch_id": manifest["batch_id"],
        "articles_sha256": articles_sha256,
        "model_version": manifest["model_version"],
        "prompt_version": manifest["prompt_version"],
        "success_count": success_count,
        "error_count": error_count,
        "started_at": started_at.isoformat(),
        "finished_at": utc_now().isoformat(),
        "results_path": str(RESULTS_PATH),
        "log_path": str(LOG_PATH),
        "progress_path": str(PROGRESS_PATH),
        "session_id": SESSION_ID,
    }

    log_progress(
        "job_report_write_started",
        path=REPORT_PATH,
        success_count=success_count,
        error_count=error_count,
    )

    REPORT_PATH.write_text(
        json.dumps(report, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    log_progress(
        "job_report_written",
        path=REPORT_PATH,
        size_bytes=REPORT_PATH.stat().st_size,
    )
    return report


## 9. Main batch runner


In [9]:
@logged_function()
def main() -> None:
    configure_runner_logging()

    started_at = utc_now()
    run_started = time.monotonic()

    log_progress(
        "enrichment_run_started",
        input_root=INPUT_ROOT,
        output_root=OUTPUT_ROOT,
        log_path=LOG_PATH,
        results_path=RESULTS_PATH,
        progress_path=PROGRESS_PATH,
        report_path=REPORT_PATH,
    )

    try:
        # Validate lại để main() vẫn an toàn nếu người dùng bỏ qua preflight cell.
        log_progress("main_stage_started", stage="preflight")
        resolved = preflight()
        log_progress("main_stage_completed", stage="preflight")

        manifest_path = resolved["manifest_path"]
        articles_path = resolved["articles_path"]
        manifest = resolved["manifest"]
        articles_sha256 = resolved["articles_sha256"]
        model_path = resolved["model_path"]

        log_progress(
            "run_inputs_ready",
            manifest=manifest_path,
            articles=articles_path,
            model=model_path,
            article_count=manifest["article_count"],
        )

        log_progress("main_stage_started", stage="model_load")
        tokenizer, model = load_model(model_path)
        log_progress("main_stage_completed", stage="model_load")

        success_count = 0
        error_count = 0

        log_progress(
            "results_file_open_started",
            path=RESULTS_PATH,
            mode="w",
        )

        with (
            articles_path.open(encoding="utf-8") as source,
            RESULTS_PATH.open("w", encoding="utf-8") as destination,
        ):
            log_progress("results_file_open_completed", path=RESULTS_PATH)

            for article_number, line in enumerate(source, start=1):
                article = json.loads(line)
                article_started = time.monotonic()

                elapsed_before = time.monotonic() - run_started
                completed_before = article_number - 1
                rate = (
                    completed_before / elapsed_before
                    if elapsed_before > 0
                    else 0.0
                )
                eta = (
                    (manifest["article_count"] - completed_before) / rate
                    if rate > 0
                    else None
                )

                progress_percent = (
                    round(
                        completed_before
                        * 100
                        / manifest["article_count"],
                        2,
                    )
                    if manifest["article_count"]
                    else 100.0
                )

                log_progress(
                    "article_started",
                    article_number=article_number,
                    article_total=manifest["article_count"],
                    progress_percent=progress_percent,
                    eta_seconds=(
                        round(eta, 2)
                        if eta is not None
                        else "n/a"
                    ),
                    article_version_id=article.get("article_version_id"),
                    input_line_chars=len(line),
                )

                try:
                    record = process_article(
                        tokenizer,
                        model,
                        article,
                        model_version=manifest["model_version"],
                        prompt_version=manifest["prompt_version"],
                    )
                    success_count += 1

                    log_progress(
                        "article_completed",
                        article_number=article_number,
                        article_total=manifest["article_count"],
                        article_version_id=article.get("article_version_id"),
                        claim_count=len(record["result"]["claims"]),
                        summary_chars=len(record["result"]["summary_en"]),
                        duration_seconds=round(
                            time.monotonic() - article_started,
                            3,
                        ),
                    )
                except Exception as error:
                    record = error_record(article, error)
                    error_count += 1

                    LOGGER.exception(
                        "article_failed session_id=%s article_number=%s "
                        "article_total=%s article_version_id=%s "
                        "error_type=%s duration_seconds=%s",
                        SESSION_ID,
                        article_number,
                        manifest["article_count"],
                        article.get("article_version_id"),
                        type(error).__name__,
                        round(time.monotonic() - article_started, 3),
                    )

                    maybe_recover_from_cuda_oom(error)

                serialized = json.dumps(
                    record,
                    ensure_ascii=False,
                    separators=(",", ":"),
                ) + "\n"

                log_event(
                    "result_record_write_started",
                    level=logging.DEBUG,
                    article_number=article_number,
                    article_version_id=article.get("article_version_id"),
                    bytes=len(serialized.encode("utf-8")),
                )

                destination.write(serialized)
                destination.flush()

                log_progress(
                    "result_record_flushed",
                    article_number=article_number,
                    article_version_id=article.get("article_version_id"),
                    status=record["status"],
                    results_file_size_bytes=RESULTS_PATH.stat().st_size,
                )

                processed_count = success_count + error_count
                elapsed = time.monotonic() - run_started

                write_progress(
                    manifest=manifest,
                    processed_count=processed_count,
                    success_count=success_count,
                    error_count=error_count,
                    current_article_version_id=article.get(
                        "article_version_id"
                    ),
                    elapsed_seconds=elapsed,
                )

                should_log_resources = (
                    article_number == 1
                    or article_number % RESOURCE_LOG_EVERY_ARTICLES == 0
                    or article_number == manifest["article_count"]
                )

                log_event(
                    "resource_log_decision",
                    level=logging.DEBUG,
                    article_number=article_number,
                    should_log=should_log_resources,
                )

                if should_log_resources:
                    log_resource_snapshot(
                        f"after_article_{article_number}"
                    )
                    log_nvidia_smi(
                        f"after_article_{article_number}"
                    )

        log_progress(
            "article_loop_completed",
            success_count=success_count,
            error_count=error_count,
            processed_count=success_count + error_count,
        )

        log_progress("main_stage_started", stage="job_report")
        write_job_report(
            manifest=manifest,
            articles_sha256=articles_sha256,
            success_count=success_count,
            error_count=error_count,
            started_at=started_at,
        )
        log_progress("main_stage_completed", stage="job_report")

        log_resource_snapshot("run_completed")
        log_nvidia_smi("run_completed")

        log_progress(
            "enrichment_run_completed",
            article_count=manifest["article_count"],
            success_count=success_count,
            error_count=error_count,
            duration_seconds=round(
                time.monotonic() - run_started,
                3,
            ),
            results_path=RESULTS_PATH,
            report_path=REPORT_PATH,
            progress_path=PROGRESS_PATH,
            log_path=LOG_PATH,
        )

    except Exception as error:
        LOGGER.exception(
            "enrichment_run_failed session_id=%s error_type=%s "
            "duration_seconds=%s",
            SESSION_ID,
            type(error).__name__,
            round(time.monotonic() - run_started, 3),
        )

        log_resource_snapshot("fatal_error")
        log_nvidia_smi("fatal_error")
        raise


## 9. Input + GPU diagnostics

Cell này cho biết:
- dataset/model có thực sự được attach dưới `/kaggle/input` hay không;
- dataset/model fallback qua KaggleHub có đang bật không;
- GPU hiện tại có compute capability nào;
- PyTorch hiện tại được build cho những CUDA architecture nào.

> Với environment hiện tại, **Tesla P100 (`sm_60`) không tương thích với PyTorch CUDA build đang có**. Hãy chọn **GPU T4 x2** trên Kaggle; runner hiện dùng `cuda:0`.


In [10]:
print("=== KAGGLE INPUT + RUNTIME DIAGNOSTICS ===")
print(f"INPUT_ROOT: {INPUT_ROOT}")
print(f"exists: {INPUT_ROOT.exists()}")

if INPUT_ROOT.exists():
    entries = sorted(INPUT_ROOT.iterdir(), key=lambda p: p.name.casefold())
    if entries:
        for entry in entries:
            print(f"- {entry}")
    else:
        print("(no attached Kaggle inputs)")
else:
    print("(input root does not exist)")

print()
print("Dataset handle:", DATASET_HANDLE or "(none)")
print("Dataset KaggleHub fallback:", ALLOW_KAGGLEHUB_DATASET_FALLBACK)
print("Model override:", MODEL_HANDLE_OVERRIDE or "(none)")
print("Model KaggleHub fallback:", ALLOW_KAGGLEHUB_MODEL_FALLBACK)

try:
    import torch

    print()
    print("Torch:", torch.__version__)
    print("Torch CUDA runtime:", torch.version.cuda)
    print("Torch CUDA architectures:", torch.cuda.get_arch_list())

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            major, minor = torch.cuda.get_device_capability(i)
            print(
                f"GPU {i}: {torch.cuda.get_device_name(i)} "
                f"(compute capability {major}.{minor}, sm_{major}{minor})"
            )
except Exception as diagnostic_error:
    print("Runtime diagnostic error:", repr(diagnostic_error))


=== KAGGLE INPUT + RUNTIME DIAGNOSTICS ===
INPUT_ROOT: /kaggle/input
exists: True
- /kaggle/input/datasets

Dataset handle: pmv259/footballpulse-ai-batches
Dataset KaggleHub fallback: True
Model override: (none)
Model KaggleHub fallback: True

Torch: 2.10.0+cu128
Torch CUDA runtime: 12.8
Torch CUDA architectures: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU 0: Tesla T4 (compute capability 7.5, sm_75)
GPU 1: Tesla T4 (compute capability 7.5, sm_75)


## 10. Preflight — chạy cell này trước Main run


In [11]:
configure_runner_logging()

try:
    info = preflight()
except Exception:
    LOGGER.exception(
        "preflight_cell_failed session_id=%s",
        SESSION_ID,
    )
    raise

print("\n=== PREFLIGHT OK ===")
print(f"Session:  {SESSION_ID}")
print(f"Manifest: {info['manifest_path']}")
print(f"Articles: {info['articles_path']}")
print(f"Model:    {info['model_path']}")
print(f"Log:      {LOG_PATH}")


2026-08-17 15:52:20,070 INFO [footballpulse.kaggle.runner] logger_configured session_id=20260817T155215Z-bb26cb6a log_level=INFO log_path=/kaggle/working/footballpulse-enrichment.log
2026-08-17 15:52:20,071 INFO [footballpulse.kaggle.runner] function_started session_id=20260817T155215Z-bb26cb6a function=preflight
2026-08-17 15:52:20,073 INFO [footballpulse.kaggle.runner] logger_configured session_id=20260817T155215Z-bb26cb6a log_level=INFO log_path=/kaggle/working/footballpulse-enrichment.log
2026-08-17 15:52:20,074 INFO [footballpulse.kaggle.runner] preflight_started session_id=20260817T155215Z-bb26cb6a input_root=/kaggle/input output_root=/kaggle/working
2026-08-17 15:52:20,076 INFO [footballpulse.kaggle.runner] preflight_stage_started session_id=20260817T155215Z-bb26cb6a stage=environment
2026-08-17 15:52:20,078 INFO [footballpulse.kaggle.runner] function_started session_id=20260817T155215Z-bb26cb6a function=log_environment
2026-08-17 15:52:31,481 INFO [footballpulse.kaggle.runner] 

## 11. Main run — xử lý toàn bộ batch


In [12]:
main()

print("\n=== RUN FINISHED ===")
print(f"Results:  {RESULTS_PATH}")
print(f"Progress: {PROGRESS_PATH}")
print(f"Report:   {REPORT_PATH}")
print(f"Log:      {LOG_PATH}")


2026-08-17 15:52:35,685 INFO [footballpulse.kaggle.runner] function_started session_id=20260817T155215Z-bb26cb6a function=main
2026-08-17 15:52:35,688 INFO [footballpulse.kaggle.runner] logger_configured session_id=20260817T155215Z-bb26cb6a log_level=INFO log_path=/kaggle/working/footballpulse-enrichment.log
2026-08-17 15:52:35,689 INFO [footballpulse.kaggle.runner] enrichment_run_started session_id=20260817T155215Z-bb26cb6a input_root=/kaggle/input output_root=/kaggle/working log_path=/kaggle/working/footballpulse-enrichment.log results_path=/kaggle/working/results.jsonl progress_path=/kaggle/working/progress.json report_path=/kaggle/working/job-report.json
2026-08-17 15:52:35,689 INFO [footballpulse.kaggle.runner] main_stage_started session_id=20260817T155215Z-bb26cb6a stage=preflight
2026-08-17 15:52:35,691 INFO [footballpulse.kaggle.runner] function_started session_id=20260817T155215Z-bb26cb6a function=preflight
2026-08-17 15:52:35,692 INFO [footballpulse.kaggle.runner] logger_conf

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


2026-08-17 15:52:46,766 INFO [footballpulse.kaggle.runner] model_weights_loaded_to_cpu session_id=20260817T155215Z-bb26cb6a duration_seconds=4.376 model_class=Qwen3ForCausalLM
2026-08-17 15:52:46,767 INFO [footballpulse.kaggle.runner] model_move_to_gpu_started session_id=20260817T155215Z-bb26cb6a device=cuda:0
2026-08-17 15:52:47,190 INFO [footballpulse.kaggle.runner] model_move_to_gpu_completed session_id=20260817T155215Z-bb26cb6a duration_seconds=0.423 device=cuda:0
2026-08-17 15:52:47,194 INFO [footballpulse.kaggle.runner] model_configuration_ready session_id=20260817T155215Z-bb26cb6a parameter_count=751632384 trainable_parameter_count=751632384 do_sample=False max_new_tokens=512
2026-08-17 15:52:47,195 INFO [footballpulse.kaggle.runner] function_started session_id=20260817T155215Z-bb26cb6a function=log_resource_snapshot
2026-08-17 15:52:47,195 INFO [footballpulse.kaggle.runner] function_started session_id=20260817T155215Z-bb26cb6a function=read_ram_stats
2026-08-17 15:52:47,197 INF

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2026-08-17 15:52:49,662 INFO [footballpulse.kaggle.runner] model_generate_completed session_id=20260817T155215Z-bb26cb6a input_tokens=719 output_tokens=30 duration_seconds=2.352 output_tokens_per_second=12.75 gpu_peak_allocated_gib=1.595
2026-08-17 15:52:49,664 INFO [footballpulse.kaggle.runner] generation_decode_completed session_id=20260817T155215Z-bb26cb6a output_chars=124 duration_seconds=0.001
2026-08-17 15:52:49,665 INFO [footballpulse.kaggle.runner] function_completed session_id=20260817T155215Z-bb26cb6a function=generate duration_seconds=2.389
2026-08-17 15:52:49,666 INFO [footballpulse.kaggle.runner] chunk_model_output_received session_id=20260817T155215Z-bb26cb6a article_version_id=956d5f07-486d-530b-9d7d-e13f3c43deaa raw_chars=124
2026-08-17 15:52:49,667 INFO [footballpulse.kaggle.runner] chunk_summary_only_completed session_id=20260817T155215Z-bb26cb6a article_version_id=956d5f07-486d-530b-9d7d-e13f3c43deaa summary_chars=124
2026-08-17 15:52:49,668 INFO [footballpulse.kaggl

## 12. Diagnostic nhanh — dùng sau khi lỗi hoặc sau khi dừng run


In [13]:
def show_runner_status(last_log_lines: int = 80) -> None:
    print("=== FOOTBALLPULSE RUNNER STATUS ===")
    print(f"Session: {SESSION_ID}")
    print(f"Log: {LOG_PATH}")
    print(f"Results: {RESULTS_PATH}")
    print(f"Progress: {PROGRESS_PATH}")
    print(f"Report: {REPORT_PATH}")

    if PROGRESS_PATH.exists():
        print("\n--- progress.json ---")
        print(PROGRESS_PATH.read_text(encoding="utf-8"))
    else:
        print("\nprogress.json chưa tồn tại.")

    if LOG_PATH.exists():
        print(f"\n--- last {last_log_lines} log lines ---")
        lines = LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        for line in lines[-last_log_lines:]:
            print(line)
    else:
        print("\nLog file chưa tồn tại.")


# Bỏ comment khi cần xem nhanh trạng thái:
# show_runner_status(80)
